# Model Train

In [5]:
import re
from collections import Counter

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

from src.model import HERANet
from src.train import MyTrain
from src.evaluate import MyEvaluate
from src.preprocessing import apply_augmentation


# =========================================================
# 0) 설정
# =========================================================
DATA_DIR = "data"

BATCH_SIZE = 16
MAX_LEN = 650
MIN_FREQ = 2
MAX_VOCAB_SIZE = 30000
VAL_SIZE = 0.1
RANDOM_STATE = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


# =========================================================
# 1) tokenizer / vocab
# =========================================================
def simple_tokenize(text: str):
    text = str(text).lower()
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)


class Vocab:
    def __init__(self, counter, min_freq=2, max_size=30000, specials=None):
        if specials is None:
            specials = ["<pad>", "<unk>"]

        self.token_to_idx = {}
        self.idx_to_token = []

        for sp in specials:
            self.token_to_idx[sp] = len(self.idx_to_token)
            self.idx_to_token.append(sp)

        tokens = [
            (tok, freq)
            for tok, freq in counter.items()
            if freq >= min_freq and tok not in self.token_to_idx
        ]
        tokens.sort(key=lambda x: (-x[1], x[0]))

        if max_size is not None:
            keep_n = max(0, max_size - len(self.idx_to_token))
            tokens = tokens[:keep_n]

        for tok, _ in tokens:
            self.token_to_idx[tok] = len(self.idx_to_token)
            self.idx_to_token.append(tok)

        self.pad_idx = self.token_to_idx["<pad>"]
        self.unk_idx = self.token_to_idx["<unk>"]

    def __len__(self):
        return len(self.idx_to_token)

    def encode(self, tokens):
        return [self.token_to_idx.get(tok, self.unk_idx) for tok in tokens]


# =========================================================
# 2) 데이터 로드
#    dataset_1: ;
#    dataset_2, dataset_3: ,
# =========================================================
def load_one_dataset(path, sep, domain_id):
    df = pd.read_csv(path, sep=sep)
    df.columns = [str(c).strip().lower() for c in df.columns]

    for col in ["title", "text", "subject", "date", "label"]:
        if col not in df.columns:
            df[col] = ""

    df["title"] = df["title"].fillna("").astype(str)
    df["text"] = df["text"].fillna("").astype(str)
    df["subject"] = df["subject"].fillna("").astype(str)
    df["date"] = df["date"].fillna("").astype(str)
    df["label"] = df["label"].fillna("").astype(str).str.strip().str.lower()

    df = df[df["label"].isin(["fake", "real"])].copy()
    df["domain_id"] = domain_id
    return df


df1 = load_one_dataset(f"{DATA_DIR}/dataset_1.csv", sep=";", domain_id=0)
df2 = load_one_dataset(f"{DATA_DIR}/dataset_2.csv", sep=",", domain_id=1)
df3 = load_one_dataset(f"{DATA_DIR}/dataset_3.csv", sep=",", domain_id=2)

df_all = pd.concat([df1, df2, df3], ignore_index=True)

print("dataset_1:", len(df1))
print("dataset_2:", len(df2))
print("dataset_3:", len(df3))
print("total:", len(df_all))
print(df_all["label"].value_counts())


# =========================================================
# 3) 텍스트 합치기
# =========================================================
def build_full_text(row):
    parts = []

    if str(row.get("title", "")).strip():
        parts.append(str(row["title"]).strip())

    if str(row.get("subject", "")).strip():
        parts.append(f"[SUBJECT] {str(row['subject']).strip()}")

    if str(row.get("date", "")).strip():
        parts.append(f"[DATE] {str(row['date']).strip()}")

    if str(row.get("text", "")).strip():
        parts.append(str(row["text"]).strip())

    return " ".join(parts).strip()


df_all["full_text"] = df_all.apply(build_full_text, axis=1)
df_all = df_all[df_all["full_text"].str.len() > 0].reset_index(drop=True)

label_map = {"fake": 0, "real": 1}
df_all["label_id"] = df_all["label"].map(label_map).astype(int)

text_len = df_all["full_text"].str.len().astype(float)
text_len_norm = (text_len - text_len.mean()) / (text_len.std() + 1e-8)

df_all["meta_len"] = text_len_norm
df_all["meta_has_title"] = (df_all["title"].str.len() > 0).astype(float)
df_all["meta_has_subject"] = (df_all["subject"].str.len() > 0).astype(float)
df_all["meta_has_date"] = (df_all["date"].str.len() > 0).astype(float)


# =========================================================
# 4) train / val split
# =========================================================
train_df, val_df = train_test_split(
    df_all,
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE,
    stratify=df_all["label_id"],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("train:", len(train_df), "val:", len(val_df))


# =========================================================
# 5) vocab 생성
# =========================================================
counter = Counter()
for text in train_df["full_text"]:
    counter.update(simple_tokenize(text))

vocab = Vocab(counter, min_freq=MIN_FREQ, max_size=MAX_VOCAB_SIZE)

print("vocab size:", len(vocab))
print("pad_idx:", vocab.pad_idx, "unk_idx:", vocab.unk_idx)


# =========================================================
# 6) dataset / collate
# =========================================================
class NewsDataset(Dataset):
    def __init__(self, df, vocab, max_len=650):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        tokens = simple_tokenize(row["full_text"])
        input_ids = self.vocab.encode(tokens)[:self.max_len]
        length = len(input_ids)

        label = int(row["label_id"])
        domain_id = int(row["domain_id"])

        meta = torch.tensor([
            float(row["meta_len"]),
            float(row["meta_has_title"]),
            float(row["meta_has_subject"]),
            float(row["meta_has_date"]),
        ], dtype=torch.float32)

        return input_ids, length, label, domain_id, meta


def collate_fn(batch):
    input_ids_list, lengths, labels, domain_ids, metas = zip(*batch)

    max_batch_len = max(lengths)
    padded = []

    for ids in input_ids_list:
        padded_ids = ids + [vocab.pad_idx] * (max_batch_len - len(ids))
        padded.append(padded_ids)

    input_ids = torch.tensor(padded, dtype=torch.long)
    lengths = torch.tensor(lengths, dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.long)
    domain_ids = torch.tensor(domain_ids, dtype=torch.long)
    meta = torch.stack(metas, dim=0)

    return input_ids, lengths, labels, domain_ids, meta


train_dataset = NewsDataset(train_df, vocab=vocab, max_len=MAX_LEN)
val_dataset = NewsDataset(val_df, vocab=vocab, max_len=MAX_LEN)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
)

print("train batches:", len(train_loader))
print("val batches:", len(val_loader))


# =========================================================
# 7) model
# =========================================================
model = HERANet(
    vocab_size=len(vocab),
    num_classes=2,
    embedding_dim=128,
    padding_idx=vocab.pad_idx,
    chunk_size=256,
    chunk_stride=224,
    token_hidden=128,
    token_layers=1,
    sent_hidden=128,
    sent_layers=1,
    dropout=0.2,
    meta_dim=4,
    meta_out=64,
    use_domain_embedding=True,
    num_domains=3,
    domain_emb_dim=32,
).to(DEVICE)


# =========================================================
# 8) train
# =========================================================
trainer = MyTrain(
    model=model,
    device=DEVICE,
    lr=1e-3,
    weight_decay=1e-2,
    max_grad_norm=5.0,
    epochs=20,
    patience=5,
    scheduler_patience=2,
    scheduler_factor=0.5,
    class_weights=None,
    augmentation_fn=apply_augmentation,
    use_augmentation=True,
    save_path=None,
)

history = trainer.fit(train_loader, val_loader)


# =========================================================
# 9) evaluate
# =========================================================
evaluator = MyEvaluate(model=model, device=DEVICE)
result = evaluator.evaluate(val_loader)
evaluator.print_summary(result)

device: cuda
dataset_1: 32470
dataset_2: 250
dataset_3: 250
total: 32970
label
fake    17764
real    15206
Name: count, dtype: int64
train: 29673 val: 3297
vocab size: 30000
pad_idx: 0 unk_idx: 1
train batches: 1855
val batches: 207


Train 1/20:   0%|          | 0/1855 [00:00<?, ?it/s]

KeyboardInterrupt: 